# Control A — the run

Governed by `preregistration/prereg_controlA.md` (signed 2026-07-28) plus **Amendments 002 and 003** (2026-07-29).

**241 conditions**: 6 strengths x (candidate + next_k + 19 random_lens + 19 random_iso), plus a clean baseline. Roughly **70 minutes** on an A100, ~18 units.

Resumable — every condition writes its own JSON and is skipped on restart. If the session dies, re-run cell 6.

### Amendment 002 — the primary eval

The prereg says "strict scoring, clean baseline 58/90 = 64.4%". `strict` was defined by *generating* `len(answer_tokens)` tokens; the harness does a **single forward pass** and cannot score a multi-token answer. 17 of 90 answers are multi-token.

Primary becomes **strict on the 73 single-token items, clean baseline 50/73 = 68.5%** — exactly `strict` on that subset, one forward pass, and consistent with the paper's own practice (`capacity.json` filters to single-token entries). First-token on all 90 is reported as a free secondary.

### Amendment 003 — a fourth band start

Stage B2 found the unspoken intermediate is already represented from **L10–12**, ten layers below the band start, and the paper's own band scaled to 36 layers is **L13–32**. So `heavy-paper` (L13–31) is added, giving four starts spanning 69% → 37% of depth.

**Pre-registered prediction:** if under-ablation is real, the effect grows monotonically as the start moves down — `heavy-late < heavy < heavy-early < heavy-paper`. A flat curve means the band start is not load-bearing, which is the stronger result.

The primary band is unchanged. Nothing was removed.

### Before cell 1

Runtime → **A100**. Caches need ~3 GB on top of the 17.6 GB model and lens.

## Cell 1 — Setup

In [ ]:
import os, sys, subprocess, torch

!pip -q install -U transformers accelerate huggingface_hub datasets

if not os.path.isdir("/content/jacobian-lens"):
    subprocess.run(["git","clone","-q","--depth","1",
                    "https://github.com/anthropics/jacobian-lens.git"],
                   cwd="/content", check=True)

os.makedirs("/content/ablation", exist_ok=True)
open("/content/ablation/__init__.py","a").close()
for p in ("/content/jacobian-lens", "/content"):
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = "/content/jacobian-lens:/content"

import importlib; importlib.invalidate_caches()
import jlens
print("jlens OK")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"{torch.cuda.get_device_name(0)}  {free/1e9:.0f}/{total/1e9:.0f} GB free")
    if total/1e9 < 30:
        print(">>> WARNING: under 30 GB. Reduce --intact-passages if caches OOM.")
else:
    print(">>> STOP. No GPU.")

## Cell 2 — Write `ablation/directions.py`

In [ ]:
%%writefile ablation/directions.py
"""Direction selection and subspace projection for J-space / R-space ablation.

Implements the direction-set half of the ablation harness. Every selector here
produces a set of residual-stream directions of a specified size; the harness
projects them out. Keeping selection separate from projection is what makes the
matched controls of proposal 4.8 cheap: same projection, different selector.

Terminology (guide 2.1): nothing here is "the workspace". These are candidate
directions until Phase 3 says otherwise.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch


# --- J-lens vector construction -------------------------------------------

def lens_vectors(
    unembed_weight: torch.Tensor, jacobian: torch.Tensor, token_ids: torch.Tensor
) -> torch.Tensor:
    """The J-lens vectors for ``token_ids`` at one layer.

    Paper §2.1 defines the J-lens vectors as the rows of ``W_U J_l``. Only the
    requested rows are materialised: the full product is ``[vocab, d_model]``
    and is far too large to hold for a real vocabulary.

    Args:
        unembed_weight: ``W_U``, shape ``[vocab, d_model]``.
        jacobian: ``J_l``, shape ``[d_model, d_model]``.
        token_ids: Shape ``[..., k]``.

    Returns:
        Shape ``[..., k, d_model]``.
    """
    rows = unembed_weight.index_select(0, token_ids.reshape(-1).to(unembed_weight.device))
    rows = rows.to(jacobian.dtype) @ jacobian
    return rows.reshape(*token_ids.shape, jacobian.shape[-1])


# --- Selectors -------------------------------------------------------------

def select_by_rank(
    lens_logits: torch.Tensor,
    k: int,
    *,
    rank_offset: int = 0,
    excluded: torch.Tensor | None = None,
) -> torch.Tensor:
    """Token ids ranked ``rank_offset .. rank_offset + k`` by lens score.

    ``rank_offset=0`` gives the top-k the paper ablates. ``rank_offset=k`` gives
    the next-k, which is the "matched but not selected" control: same lens, same
    size, adjacent rank band. A candidate subspace that matters no more than the
    next-k has not earned H1 (proposal 4.8, extended per the probe-swap design).

    Args:
        lens_logits: Shape ``[n_positions, vocab]``.
        k: Number of directions.
        rank_offset: Rank to start from.
        excluded: Boolean mask ``[n_positions, vocab]``; True entries are never
            selected. This carries the clean-pass exclusion — see
            :func:`clean_top_mask`.

    Returns:
        Shape ``[n_positions, k]``.
    """
    scores = lens_logits.clone()
    if excluded is not None:
        scores = scores.masked_fill(excluded, float("-inf"))
    top = scores.topk(rank_offset + k, dim=-1).indices
    return top[:, rank_offset:]


def random_lens_tokens(
    n_positions: int,
    k: int,
    vocab_size: int,
    generator: torch.Generator,
    *,
    excluded: torch.Tensor | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Uniformly random token ids — matched-size random control (proposal 4.8).

    Drawn from the lens dictionary rather than isotropically, so the control
    asks "is it *these* lens directions, or any lens directions?". Strictly the
    harder question of the two; run both.
    """
    out = torch.empty(n_positions, k, dtype=torch.long, device=device)
    for p in range(n_positions):
        while True:
            cand = torch.randint(
                vocab_size, (k,), generator=generator, device=generator.device
            ).to(device)
            if excluded is None or not bool(excluded[p, cand].any()):
                out[p] = cand
                break
    return out


def random_isotropic(
    n_positions: int, k: int, d_model: int, generator: torch.Generator,
    *, device: torch.device | None = None, dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    """Isotropic random unit directions — the paper's random-direction control."""
    v = torch.randn(
        n_positions, k, d_model, generator=generator, device=generator.device,
        dtype=torch.float32,
    ).to(device=device, dtype=dtype)
    return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def clean_top_mask(
    clean_next_token_logits: torch.Tensor, n_exclude: int = 10
) -> torch.Tensor:
    """Mask marking the clean pass's top-``n_exclude`` predictions per position.

    **This is the confound guard, and it is not optional.** Paper: "we do not
    ablate any tokens that appear in the top-10 tokens of a clean forward pass,
    so as to specifically target the J-space's effects on internal reasoning
    rather than report." Without it, ablation suppresses whatever the model was
    about to say, performance drops for a trivial reason, and H1 gets
    "confirmed" by an artifact.

    Args:
        clean_next_token_logits: Shape ``[n_positions, vocab]`` from an
            unablated forward pass.

    Returns:
        Boolean ``[n_positions, vocab]``, True where a token must not be ablated.
    """
    mask = torch.zeros_like(clean_next_token_logits, dtype=torch.bool)
    top = clean_next_token_logits.topk(n_exclude, dim=-1).indices
    return mask.scatter(-1, top, True)


# --- Projection ------------------------------------------------------------

@dataclass(frozen=True)
class Basis:
    """An orthonormal-row basis with a validity mask for rank-deficient sets."""

    rows: torch.Tensor   # [n_positions, k, d_model], orthonormal rows
    keep: torch.Tensor   # [n_positions, k], float 1/0
    rank: torch.Tensor   # [n_positions], effective rank actually removed


def orthonormalise(vectors: torch.Tensor, *, rtol: float = 1e-6) -> Basis:
    """Orthonormal basis for the span of each position's direction set.

    J-lens vectors are overcomplete and non-orthogonal (paper §2.3), so a set of
    k of them may span fewer than k dimensions. Small singular values are masked
    out rather than dropped, which keeps the operation batched and makes the
    effective rank observable — report it, because "we ablated k directions" is
    false if the span was smaller.
    """
    vectors = vectors.to(torch.float32)
    _, s, vh = torch.linalg.svd(vectors, full_matrices=False)
    keep = (s > rtol * s[..., :1].clamp_min(1e-30)).to(vectors.dtype)
    return Basis(rows=vh, keep=keep, rank=keep.sum(-1))


def project_out(
    hidden: torch.Tensor, basis: Basis, *, mode: str = "subspace"
) -> torch.Tensor:
    """Remove the component of ``hidden`` inside the spanned subspace.

    Args:
        hidden: Shape ``[n_positions, d_model]``.
        mode: ``"subspace"`` projects onto the orthogonal complement of the span
            in one step. ``"sequential"`` removes each direction in turn, which
            is order-dependent for non-orthogonal vectors and therefore removes
            *less* than the full span.

    The paper's phrasing — "zero out the residual stream's projection onto
    each" — does not disambiguate these, and for non-orthogonal J-lens vectors
    they differ. ``"subspace"`` is the default because it is the one that
    actually removes the content; ``"sequential"`` is provided so the choice can
    be tested rather than assumed. Record which was used.
    """
    h = hidden.to(torch.float32)
    if mode == "subspace":
        coeffs = torch.einsum("prd,pd->pr", basis.rows, h) * basis.keep
        return (h - torch.einsum("pr,prd->pd", coeffs, basis.rows)).to(hidden.dtype)
    if mode == "sequential":
        for i in range(basis.rows.shape[1]):
            v = basis.rows[:, i, :] * basis.keep[:, i : i + 1]
            h = h - (h * v).sum(-1, keepdim=True) * v
        return h.to(hidden.dtype)
    raise ValueError(f"unknown mode {mode!r}")

## Cell 3 — Write `ablation/harness.py`

In [ ]:
%%writefile ablation/harness.py
"""Two-pass ablation harness.

Pass 1 is a clean forward pass: it records the residual stream at every band
layer, computes lens logits, and captures the clean next-token distribution.
Pass 2 re-runs with the selected directions projected out.

Two passes are not an optimisation choice — the confound guard of proposal 4.4
(paper: exclude the clean pass's top-10) *requires* knowing the clean output
before choosing what to ablate.

This harness is built to Phase 3 requirements from the first line, per guide
§3a: Control A, Control B, and the Phase 3 sweep all run through it unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Literal, Sequence

import torch

from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens

from .directions import (
    Basis,
    clean_top_mask,
    lens_vectors,
    orthonormalise,
    project_out,
    random_isotropic,
    random_lens_tokens,
    select_by_rank,
)

Selector = Literal["topk", "next_k", "random_lens", "random_iso", "none"]


def record_at_or(spec, final):
    return sorted({*spec.layers, final})


def _mask_from_ids(ids: torch.Tensor, shape) -> torch.Tensor:
    """Rebuild the clean-top-k boolean mask from cached ids."""
    m = torch.zeros(shape, dtype=torch.bool, device=ids.device)
    return m.scatter(-1, ids, True)


@dataclass(frozen=True)
class AblationSpec:
    """One fully-specified ablation condition. Serialise this into every result.

    Attributes:
        layers: Band of block indices to ablate at. Light/medium/heavy differ
            only here — the paper varies the layer range, not k.
        k: Directions removed per position (proposal 4.7 sweeps this; the paper
            fixed it at 10). Sweeping k *and* layers multiplies runs — state
            which axis in prereg_phase3.md.
        selector: Which directions. ``"none"`` is the clean baseline.
        seed: Required for every random selector (guide §1.2).
        exclude_clean_top: Confound guard size. **Do not set to 0** except as a
            deliberate, logged demonstration of the artifact it prevents.
        mode: Projection mode; see :func:`project_out`.
        positions: Token positions to ablate at; ``None`` means all.
    """

    layers: tuple[int, ...]
    k: int
    selector: Selector = "topk"
    seed: int | None = None
    exclude_clean_top: int = 10
    mode: str = "subspace"
    positions: tuple[int, ...] | None = None

    def __post_init__(self) -> None:
        if self.selector in ("random_lens", "random_iso") and self.seed is None:
            raise ValueError(
                "random selectors require an explicit seed — an unseeded "
                "matched-random baseline is not reproducible and proposal 4.8 "
                "results computed against it are not reportable"
            )

    def key(self) -> str:
        import hashlib, json
        blob = json.dumps(asdict(self), sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:16]


@dataclass
class AblationResult:
    logits: torch.Tensor              # [n_positions, vocab] ablated next-token logits
    clean_logits: torch.Tensor        # [n_positions, vocab] unablated
    effective_rank: dict[int, torch.Tensor] = field(default_factory=dict)
    spec: AblationSpec | None = None
    ids: torch.Tensor | None = None       # [1, seq_len]; needed to score against
                                          # true next tokens on a corpus (intact side)


def prepare_lens(lens: JacobianLens, device) -> JacobianLens:
    """Move the Jacobians onto the compute device. **Does not touch dtype.**

    ``JacobianLens.__init__`` does ``J.float()`` on every Jacobian, so the class
    holds float32 regardless of what was on disk (``save`` writes fp16 purely
    for compactness). ``apply()`` correspondingly casts residuals with
    ``.float()`` before ``transport``. The library's internal contract is
    float32 throughout, and ``HFLensModel.unembed`` casts to the head's dtype
    itself, so nothing downstream needs the model's dtype here.

    An earlier version of this function cast the Jacobians to the model's dtype.
    That broke the contract and produced
    ``RuntimeError: expected mat1 and mat2 to have the same dtype`` inside
    ``transport``. Callers passing activations straight from
    ``ActivationRecorder`` (which are in the *model's* dtype, not float32) must
    cast those to float — see :func:`build_cache`.

    Only the device move is needed: without it ``transport`` copies a
    ``[d_model, d_model]`` matrix host-to-device on every call.
    """
    lens.jacobians = {k: v.to(device=device) for k, v in lens.jacobians.items()}
    return lens


@dataclass
class PromptCache:
    """Per-prompt work that every condition would otherwise repeat.

    The clean forward pass and the lens readout are identical across all
    conditions for a given prompt — only the direction *selection* differs. A
    37-condition sweep without this recomputes both 37 times.

    What is cached is deliberately small: the ranked token ids per layer, not
    the lens logits themselves. Lens logits are ``[n_positions, vocab]``, which
    at a 150k vocabulary is megabytes per layer per prompt; the ranked ids are
    ``[n_positions, k_max]``. Any ``k <= k_max`` is then a slice.

    ``k_max`` must be at least ``2 * max(k)`` in the sweep, because the
    ``next_k`` selector reads ranks ``k..2k``.
    """

    ids: torch.Tensor
    n_pos: int
    clean_logits: torch.Tensor
    excluded_ids: torch.Tensor | None          # [n_pos, n_exclude]
    ranked_ids: dict[int, torch.Tensor]        # layer -> [n_pos, k_max]
    k_max: int


@torch.no_grad()
def build_cache(
    model: Any, lens: JacobianLens, prompt: str, layers: Sequence[int],
    *, k_max: int, exclude_clean_top: int = 10, max_seq_len: int = 512,
) -> PromptCache:
    """Run the clean pass once and rank directions once, for reuse."""
    ids = model.encode(prompt, max_length=max_seq_len)
    final = model.n_layers - 1
    record_at = sorted({*layers, final})

    with ActivationRecorder(model.layers, record_at) as rec:
        model.forward(ids)
        acts = {i: rec.activations[i][0].detach() for i in record_at}
    clean_logits = model.unembed(acts[final])

    excluded = (clean_top_mask(clean_logits, exclude_clean_top)
                if exclude_clean_top > 0 else None)
    excluded_ids = (clean_logits.topk(exclude_clean_top, dim=-1).indices
                    if exclude_clean_top > 0 else None)

    ranked = {}
    for layer in layers:
        # .float() to match the lens's float32 Jacobians, as apply() does.
        lens_logits = model.unembed(lens.transport(acts[layer].float(), layer))
        ranked[layer] = select_by_rank(lens_logits, k_max, excluded=excluded)

    return PromptCache(ids, ids.shape[1], clean_logits, excluded_ids, ranked, k_max)


class _Ablator:
    """Forward hooks that project out precomputed per-position direction sets."""

    def __init__(
        self,
        blocks: Sequence[torch.nn.Module],
        bases: dict[int, Basis],
        position_mask: torch.Tensor | None,
        mode: str,
    ) -> None:
        self._blocks, self._bases = blocks, bases
        self._position_mask, self._mode = position_mask, mode
        self._handles: list[Any] = []

    def _hook(self, index: int):
        basis = self._bases[index]

        def fn(module, inputs, output):
            is_tuple = not torch.is_tensor(output)
            tensor = output[0] if is_tuple else output
            # tensor: [batch, seq, d_model]; harness runs batch=1.
            h = tensor[0]
            new = project_out(h, basis, mode=self._mode)
            if self._position_mask is not None:
                new = torch.where(self._position_mask[:, None], new, h)
            tensor = torch.cat([new[None], tensor[1:]], dim=0)
            return (tensor, *output[1:]) if is_tuple else tensor

        return fn

    def __enter__(self):
        try:
            for i in self._bases:
                self._handles.append(self._blocks[i].register_forward_hook(self._hook(i)))
        except Exception:
            self.__exit__()
            raise
        return self

    def __exit__(self, *exc) -> None:
        for h in self._handles:
            h.remove()
        self._handles = []


@torch.no_grad()
def run_ablation(
    model: Any,
    lens: JacobianLens,
    unembed_weight: torch.Tensor,
    prompt: str,
    spec: AblationSpec,
    *,
    max_seq_len: int = 512,
    cache: PromptCache | None = None,
) -> AblationResult:
    """Run one ablation condition end to end.

    Args:
        model: Anything satisfying ``jlens.protocol.LensModel``.
        lens: Fitted lens. ``spec.layers`` must be a subset of its source layers
            for lens-based selectors.
        unembed_weight: ``W_U``, ``[vocab, d_model]`` — usually
            ``model.lm_head.weight``. Passed explicitly because the LensModel
            protocol exposes ``unembed()`` (norm + head) but not ``W_U`` itself.
    """
    final = model.n_layers - 1

    # --- Pass 1: clean (skipped entirely when a cache is supplied) ---
    if cache is not None:
        if spec.k * (2 if spec.selector == "next_k" else 1) > cache.k_max:
            raise ValueError(
                f"cache holds k_max={cache.k_max} ranked directions but this "
                f"condition needs {spec.k * (2 if spec.selector == 'next_k' else 1)}. "
                "Rebuild the cache with a larger k_max."
            )
        ids, n_pos = cache.ids, cache.n_pos
        clean_logits, acts = cache.clean_logits, None
    else:
        ids = model.encode(prompt, max_length=max_seq_len)
        n_pos = ids.shape[1]
        with ActivationRecorder(model.layers, sorted({*spec.layers, final})) as rec:
            model.forward(ids)
            acts = {i: rec.activations[i][0].detach() for i in record_at_or(spec, final)}
        clean_logits = model.unembed(acts[final])

    if spec.selector == "none":
        return AblationResult(clean_logits, clean_logits, spec=spec, ids=ids)

    excluded = None
    if spec.exclude_clean_top > 0:
        excluded = (
            _mask_from_ids(cache.excluded_ids, clean_logits.shape)
            if cache is not None
            else clean_top_mask(clean_logits, spec.exclude_clean_top)
        )

    # --- Direction selection, per band layer ---
    gen = torch.Generator(device="cpu")
    if spec.seed is not None:
        gen.manual_seed(spec.seed)
    bases: dict[int, Basis] = {}
    ref = clean_logits
    for layer in spec.layers:
        h = acts[layer] if acts is not None else ref
        if spec.selector == "random_iso":
            vecs = random_isotropic(
                n_pos, spec.k, model.d_model, gen, device=h.device, dtype=h.dtype
            )
        else:
            ranked = cache.ranked_ids[layer] if cache is not None else None
            if ranked is None:
                lens_logits = model.unembed(lens.transport(h.float(), layer))
            if spec.selector == "topk":
                tok = (ranked[:, : spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k, excluded=excluded))
            elif spec.selector == "next_k":
                tok = (ranked[:, spec.k : 2 * spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k,
                                           rank_offset=spec.k, excluded=excluded))
            elif spec.selector == "random_lens":
                tok = random_lens_tokens(
                    n_pos, spec.k, clean_logits.shape[-1], gen,
                    excluded=excluded, device=clean_logits.device,
                )
            else:
                raise ValueError(f"unknown selector {spec.selector!r}")
            vecs = lens_vectors(unembed_weight, lens.jacobians[layer].to(h.device), tok)
        bases[layer] = orthonormalise(vecs)

    pos_mask = None
    if spec.positions is not None:
        pos_mask = torch.zeros(n_pos, dtype=torch.bool, device=ids.device)
        pos_mask[list(spec.positions)] = True

    # --- Pass 2: ablated ---
    with _Ablator(model.layers, bases, pos_mask, spec.mode):
        with ActivationRecorder(model.layers, [final]) as rec2:
            model.forward(ids)
            ablated_final = rec2.activations[final][0].detach()

    return AblationResult(
        logits=model.unembed(ablated_final),
        clean_logits=clean_logits,
        effective_rank={l: b.rank for l, b in bases.items()},
        spec=spec,
        ids=ids,
    )


def greedy_match(result: AblationResult, answer_id: int, position: int = -1) -> dict:
    """Score one prompt: did the greedy next token match, clean and ablated?

    The Control A metric per DECISION_control_A §4.4 — greedy next-token
    accuracy against probe-swap.json's ``answer`` field.
    """
    return {
        "clean_correct": int(result.clean_logits[position].argmax()) == answer_id,
        "ablated_correct": int(result.logits[position].argmax()) == answer_id,
    }

## Cell 4 — Write `ablation/intact.py`

In [ ]:
%%writefile ablation/intact.py
"""Stage C2 — the intact side of Control A.

`DECISION_control_A.md` §4.5. This half is not optional and is not secondary.

The paper's claim is **selective** degradation: ablating the J-space collapses
multi-step reasoning while leaving fluent generation and unrelated tasks
comparatively intact. A degrading-side measurement alone cannot distinguish

    "we removed the workspace"      from      "we broke the model"

because both produce a large drop on the reasoning eval. Only the contrast
between the two sides separates them.

Primary measure, following the paper: **top-1 match on a pretraining-like
corpus** — the fraction of positions where the ablated model's most likely next
token agrees with the unablated model's. WikiText is used because it needs no
extra download (jlens ships a loader), it is the corpus the published lens was
fitted on, and next-token prediction has no floor effect, so "intact" cannot be
trivially satisfied the way a chance-level classification task could be.

Two further measures come from the same forward passes at no extra cost, and are
reported because top-1 match is coarse — it registers nothing until the argmax
actually flips:

  top5_overlap   graded agreement; catches reordering below the argmax
  mean_kl        KL(clean || ablated), the full distributional shift

Cross-entropy against the *true* next token is also reported when ids are
available, since "the ablated model still predicts real text well" is a stronger
claim than "it agrees with itself".
"""

from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Any, Sequence

import torch

from .harness import AblationResult, AblationSpec, PromptCache, build_cache, run_ablation


@dataclass
class IntactResult:
    """Intact-side metrics for one prompt under one ablation condition."""

    top1_match: float          # 1.0 = ablation never changed the argmax
    top5_overlap: float        # mean |top5_clean ∩ top5_ablated| / 5
    mean_kl: float             # mean KL(clean || ablated), nats
    clean_ce: float | None     # cross-entropy vs true next token, unablated
    ablated_ce: float | None   # same, ablated
    n_positions: int

    @property
    def ce_delta(self) -> float | None:
        """Increase in cross-entropy caused by ablation. The headline number if
        ids were available: how much worse the model predicts real text."""
        if self.clean_ce is None or self.ablated_ce is None:
            return None
        return self.ablated_ce - self.clean_ce


@torch.no_grad()
def score_intact(result: AblationResult, *, skip_first: int = 4) -> IntactResult:
    """Score one ablation result on the intact side.

    Args:
        result: from :func:`run_ablation`, on a corpus prompt.
        skip_first: positions dropped from the front. Early positions are
            attention sinks with atypical residual statistics; kept low because
            §A.7 found position masking gave no meaningful improvement, so
            there is no reason to inherit the code default of 16.
    """
    clean = result.clean_logits[skip_first:].float()
    ablated = result.logits[skip_first:].float()
    n = clean.shape[0]
    if n == 0:
        return IntactResult(float("nan"), float("nan"), float("nan"), None, None, 0)

    top1_match = float((clean.argmax(-1) == ablated.argmax(-1)).float().mean())

    tc = clean.topk(5, dim=-1).indices
    ta = ablated.topk(5, dim=-1).indices
    overlap = (tc.unsqueeze(-1) == ta.unsqueeze(-2)).any(-1).float().sum(-1) / 5.0
    top5_overlap = float(overlap.mean())

    logp_c = torch.log_softmax(clean, dim=-1)
    logp_a = torch.log_softmax(ablated, dim=-1)
    mean_kl = float((logp_c.exp() * (logp_c - logp_a)).sum(-1).mean())

    clean_ce = ablated_ce = None
    if result.ids is not None:
        # position i predicts token i+1; drop the final position, which has no target
        targets = result.ids[0][skip_first + 1:]
        m = min(targets.shape[0], n - 1)
        if m > 0:
            t = targets[:m]
            clean_ce = float(torch.nn.functional.cross_entropy(clean[:m], t))
            ablated_ce = float(torch.nn.functional.cross_entropy(ablated[:m], t))

    return IntactResult(top1_match, top5_overlap, mean_kl, clean_ce, ablated_ce, n)


def aggregate(results: Sequence[IntactResult]) -> dict[str, Any]:
    """Mean each metric across prompts, ignoring empty results."""
    rs = [r for r in results if r.n_positions > 0]
    if not rs:
        return {"n_prompts": 0}
    out: dict[str, Any] = {"n_prompts": len(rs),
                           "n_positions": sum(r.n_positions for r in rs)}
    for f in ("top1_match", "top5_overlap", "mean_kl"):
        out[f] = sum(getattr(r, f) for r in rs) / len(rs)
    for f in ("clean_ce", "ablated_ce"):
        vals = [getattr(r, f) for r in rs if getattr(r, f) is not None]
        out[f] = sum(vals) / len(vals) if vals else None
    if out.get("clean_ce") is not None and out.get("ablated_ce") is not None:
        out["ce_delta"] = out["ablated_ce"] - out["clean_ce"]
    return out


@torch.no_grad()
def run_intact_side(
    model: Any, lens: Any, unembed_weight: torch.Tensor,
    corpus_prompts: Sequence[str], spec: AblationSpec,
    *, skip_first: int = 4, max_seq_len: int = 128, k_max: int | None = None,
    verbose: bool = True,
) -> dict[str, Any]:
    """Run one ablation condition across a corpus and aggregate the intact metrics."""
    k_max = k_max or max(2 * spec.k, 1)
    results = []
    for i, prompt in enumerate(corpus_prompts):
        cache = (build_cache(model, lens, prompt, spec.layers, k_max=k_max,
                             exclude_clean_top=spec.exclude_clean_top,
                             max_seq_len=max_seq_len)
                 if spec.selector != "none" else None)
        r = run_ablation(model, lens, unembed_weight, prompt, spec,
                         cache=cache, max_seq_len=max_seq_len)
        results.append(score_intact(r, skip_first=skip_first))
        if verbose and (i + 1) % 5 == 0:
            print(f"  [{i+1}/{len(corpus_prompts)}] "
                  f"top1_match={results[-1].top1_match:.3f}")
    return aggregate(results)


# --- the diagnostic that stops a vacuous "intact" -------------------------

NO_EFFECT_TOP1 = 0.99
NO_EFFECT_KL = 1e-3

#: Above this, the corpus counts as preserved. Exposed as a parameter of
#: :func:`diagnose` so the value used is recorded rather than buried — it is a
#: judgment call, not a fact, and the paper gives no numeric criterion.
PRESERVED_TOP1 = 0.90


def diagnose(intact: dict[str, Any], degrading_drop: float | None = None,
             *, preserved_top1: float = PRESERVED_TOP1) -> dict[str, Any]:
    """Distinguish selective degradation from the two ways it can be faked.

    **The failure this exists to catch:** a top-1 match near 1.0 reads as "the
    intact side held up", but it is equally consistent with the ablation having
    done nothing at all — a mis-specified band, a silently no-op confound guard,
    a hook that never fired. In that case the degrading side must also show
    nothing, and reporting "intact preserved" would be describing a broken
    experiment as a result.

    So the intact side is only evidence *given* that the ablation demonstrably
    perturbed the model somewhere. That is why ``degrading_drop`` belongs here.

    Args:
        intact: output of :func:`aggregate`.
        degrading_drop: accuracy lost on the reasoning eval under the same
            condition, as a fraction. Pass it whenever it is known.
    """
    t1, kl = intact.get("top1_match"), intact.get("mean_kl")
    d: dict[str, Any] = {"top1_match": t1, "mean_kl": kl,
                         "degrading_drop": degrading_drop}

    no_effect = (t1 is not None and t1 >= NO_EFFECT_TOP1
                 and kl is not None and kl <= NO_EFFECT_KL)

    if no_effect and (degrading_drop is None or degrading_drop < 0.05):
        d["verdict"] = "NO EFFECT — not evidence of selectivity"
        d["reading"] = (
            "The ablation barely perturbed the corpus distribution and did not "
            "meaningfully harm the reasoning eval either. This is consistent "
            "with the ablation not being applied: check that hooks fired, that "
            "the band indexes real layers, and that the clean-top-k guard is "
            "not excluding every candidate direction. Do not report this as an "
            "intact side."
        )
    elif no_effect:
        d["verdict"] = "SUSPICIOUS — large reasoning drop with no corpus shift"
        d["reading"] = (
            "The reasoning eval dropped substantially while the corpus "
            "distribution is essentially unchanged. That is a stronger "
            "selectivity claim than the paper makes, and warrants checking that "
            "the reasoning drop is not an artifact of the eval or the guard "
            "before it is believed."
        )
    elif degrading_drop is not None and degrading_drop >= 0.05:
        # BOTH conditions are required. An earlier version tested only for
        # degradation and labelled heavy corpus disruption "SELECTIVE" — the
        # exact conflation this module exists to prevent.
        corpus_disruption = 1.0 - (t1 if t1 is not None else 0.0)
        d["corpus_disruption"] = corpus_disruption
        d["selectivity_ratio"] = degrading_drop / max(corpus_disruption, 1e-6)
        if t1 is not None and t1 >= preserved_top1:
            d["verdict"] = "SELECTIVE — degradation with corpus largely preserved"
            d["reading"] = (
                f"Reasoning fell by {degrading_drop:.1%} while top-1 agreement on "
                f"the corpus held at {t1:.1%} (threshold {preserved_top1:.0%}). "
                f"Selectivity ratio {d['selectivity_ratio']:.1f}x. This is the "
                "paper's claim shape. Compare against the matched random-subspace "
                "condition before attributing it to the candidate subspace "
                "(proposal 4.8)."
            )
        else:
            d["verdict"] = "DAMAGE — both sides degraded"
            d["reading"] = (
                f"Reasoning fell by {degrading_drop:.1%}, but corpus top-1 "
                f"agreement also fell to {t1:.1%}, below the {preserved_top1:.0%} "
                "preservation threshold. Ablation degraded the model generally "
                "rather than selectively, and the workspace interpretation is "
                "not supported by this condition regardless of how large the "
                "reasoning drop was."
            )
    else:
        d["verdict"] = "INCONCLUSIVE"
        d["reading"] = (
            "The corpus distribution moved but no reasoning drop was supplied, "
            "or it was below 5%. Selectivity cannot be assessed from one side."
        )
    return d

## Cell 5 — Write `run_control_a.py`

In [ ]:
%%writefile run_control_a.py
"""Control A — the full run. Degrading side + intact side + criteria.

Governed by `preregistration/prereg_controlA.md`, signed 2026-07-28.
Implements `DECISION_control_A.md`.

Nothing here decides whether Control A passed. It computes the pre-registered
quantities and prints them against the pre-registered thresholds. The call is
the user's (AI collaboration guide §2, row 12).

Usage:
    python run_control_a.py \
        --model Qwen/Qwen3-8B \
        --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
        --data jacobian-lens/data/experiments/probe-swap.json \
        --out results/raw/controlA_qwen3-8b/
"""
from __future__ import annotations

import argparse, json, subprocess, time
from collections import defaultdict
from dataclasses import asdict
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens.examples import load_wikitext_prompts
from jlens.lens import JacobianLens
from ablation.harness import AblationSpec, build_cache, prepare_lens, run_ablation
from ablation.intact import aggregate, diagnose, score_intact

LENS_REPO = "neuronpedia/jacobian-lens"

# --- prereg §2 and §3. Shared start; heavy additionally at two other starts. ---
STRENGTHS: dict[str, tuple[int, ...]] = {
    "light":       tuple(range(20, 24)),   # 4 layers
    "medium":      tuple(range(20, 28)),   # 8
    "heavy":       tuple(range(20, 32)),   # 12  <- primary
    "heavy-early": tuple(range(15, 32)),   # 17  sensitivity: under-ablation
    "heavy-late":  tuple(range(24, 32)),   # 8   sensitivity: over-ablation
    "heavy-paper": tuple(range(13, 32)),   # 19  Amendment 003
}

#: Band-start dose-response, ordered from the highest start to the lowest
#: (69% -> 37% of depth). Amendment 003 pre-registers the prediction that IF
#: under-ablation is real, the effect grows monotonically down this list; a flat
#: curve means the band start is not load-bearing.
START_ORDER = ("heavy-late", "heavy", "heavy-early", "heavy-paper")
PRIMARY = "heavy"

# prereg §4: relative reduction, dose-response gap, intact-side floor
REL_REDUCTION_REQUIRED = 0.50   # prereg §4; Amendment 002 moves the absolute
                                # target from <=32.2% to <=34.3% (clean 68.5%)
DOSE_GAP_PP = 10.0
PRESERVED_TOP1 = 0.90


def norm(prompt: str, answer: str) -> tuple[str, str]:
    """The trailing-whitespace fix: 29 of 90 prompts end with a space."""
    return prompt.rstrip(), " " + answer.strip()


def git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "UNKNOWN"


def build_grid(n_draws: int, base_seed: int) -> list[tuple[str, AblationSpec]]:
    """Candidate + matched controls at every strength (prereg §4.1: 19 draws)."""
    grid: list[tuple[str, AblationSpec]] = [
        ("clean", AblationSpec(layers=(), k=0, selector="none"))
    ]
    for name, layers in STRENGTHS.items():
        grid.append((f"{name}|topk", AblationSpec(layers=layers, k=10, selector="topk")))
        grid.append((f"{name}|next_k", AblationSpec(layers=layers, k=10, selector="next_k")))
        for d in range(n_draws):
            seed = base_seed + 1000 * d + len(layers)
            grid.append((f"{name}|random_lens|{d}",
                         AblationSpec(layers=layers, k=10, selector="random_lens", seed=seed)))
            grid.append((f"{name}|random_iso|{d}",
                         AblationSpec(layers=layers, k=10, selector="random_iso", seed=seed)))
    return grid


@torch.no_grad()
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--lens-file", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--n-draws", type=int, default=19)      # prereg §4.1
    ap.add_argument("--base-seed", type=int, default=20260728)
    ap.add_argument("--intact-passages", type=int, default=20)
    ap.add_argument("--intact-random-sample", type=int, default=5,
                    help="random draws given the intact side, at the primary band only. "
                         "The intact side exists to establish SELECTIVITY of the "
                         "candidate; running it on all 19x5 random draws would "
                         "roughly double runtime for little added evidence.")
    ap.add_argument("--max-seq-len", type=int, default=128)
    ap.add_argument("--skip-first", type=int, default=4)
    args = ap.parse_args()

    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = getattr(torch, args.dtype)

    print(f"loading {args.model} ...")
    hf = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, device_map=device)
    tok = AutoTokenizer.from_pretrained(args.model)
    lm = jlens.from_hf(hf, tok)
    lens = prepare_lens(JacobianLens.from_pretrained(LENS_REPO, filename=args.lens_file), device)
    wu = lm._lm_head.weight.detach()
    print(f"  n_layers={lm.n_layers} d_model={lm.d_model}")

    # ---- data ------------------------------------------------------------
    items = json.load(open(args.data))["items"]
    for it in items:
        p, a = norm(it["prompt"], it["answer"])
        it["_prompt"], it["_answer"] = p, a
        it["_ans_ids"] = tok(a, add_special_tokens=False).input_ids
        it["_single"] = len(it["_ans_ids"]) == 1
    single = [it for it in items if it["_single"]]
    print(f"  {len(items)} prompts; {len(single)} with single-token answers "
          f"(primary eval, prereg amendment 002)")

    passages = load_wikitext_prompts(args.intact_passages)
    print(f"  {len(passages)} WikiText passages for the intact side")

    # ---- caches: build once, reuse across all conditions ------------------
    all_layers = tuple(sorted({l for ls in STRENGTHS.values() for l in ls}))
    print(f"\nbuilding caches over layers {all_layers[0]}..{all_layers[-1]} ...")
    t0 = time.perf_counter()
    deg_cache = {it["name"]: build_cache(lm, lens, it["_prompt"], all_layers,
                                         k_max=20, max_seq_len=args.max_seq_len)
                 for it in items}
    int_cache = [build_cache(lm, lens, p, all_layers, k_max=20,
                             max_seq_len=args.max_seq_len) for p in passages]
    print(f"  {time.perf_counter()-t0:.0f}s")

    grid = build_grid(args.n_draws, args.base_seed)
    intact_for = {f"{s}|topk" for s in STRENGTHS} | {f"{s}|next_k" for s in STRENGTHS} | {"clean"}
    intact_for |= {f"{PRIMARY}|random_lens|{d}" for d in range(args.intact_random_sample)}
    intact_for |= {f"{PRIMARY}|random_iso|{d}" for d in range(args.intact_random_sample)}
    print(f"\n{len(grid)} conditions; intact side on {len(intact_for)} of them\n")

    commit = git_commit()
    results: dict[str, dict] = {}

    for i, (name, spec) in enumerate(grid, 1):
        f = out / f"{name.replace('|','__')}.json"
        if f.exists():
            results[name] = json.loads(f.read_text()); print(f"[skip] {name}"); continue

        t = time.perf_counter()
        by_cat, strict_hits, first_hits, strict_n = defaultdict(list), 0, 0, 0
        for it in items:
            r = run_ablation(lm, lens, wu, it["_prompt"], spec,
                             cache=deg_cache[it["name"]], max_seq_len=args.max_seq_len)
            pred = int(r.logits[-1].argmax())
            first = pred == it["_ans_ids"][0]
            first_hits += first
            if it["_single"]:
                strict = tok.decode([pred]).strip().lower() == it["_answer"].strip().lower()
                strict_hits += strict; strict_n += 1
                by_cat[it["category"]].append(strict)

        rec = {
            "condition": name, "spec": asdict(spec), "git_commit": commit,
            "strict_single": {"k": strict_hits, "n": strict_n, "acc": strict_hits / strict_n},
            "first_token_all": {"k": first_hits, "n": len(items), "acc": first_hits / len(items)},
            "by_category": {c: {"n": len(v), "acc": sum(v) / len(v)}
                            for c, v in sorted(by_cat.items()) if len(v) >= 4},
            "seconds": round(time.perf_counter() - t, 1),
        }

        if name in intact_for:
            rec["intact"] = aggregate([
                score_intact(run_ablation(lm, lens, wu, p, spec, cache=c,
                                          max_seq_len=args.max_seq_len),
                             skip_first=args.skip_first)
                for p, c in zip(passages, int_cache)])

        f.write_text(json.dumps(rec, indent=2))
        results[name] = rec
        it1 = rec.get("intact", {}).get("top1_match")
        print(f"[{i:>3}/{len(grid)}] {name:<28} strict={rec['strict_single']['acc']:.3f}"
              + (f"  intact_top1={it1:.3f}" if it1 is not None else "")
              + f"  ({rec['seconds']}s)")

    (out / "_all_results.json").write_text(json.dumps(results, indent=2))
    report(results, args, out)


def report(results: dict, args, out: Path) -> None:
    """Print the pre-registered quantities against the pre-registered thresholds."""
    clean = results["clean"]["strict_single"]["acc"]
    print("\n" + "=" * 74)
    print(f"CONTROL A — clean strict (single-token subset): {clean:.1%}")
    print("=" * 74)
    print(f"{'strength':<14}{'candidate':>10}{'drop':>8}{'rel':>8}"
          f"{'next_k':>9}{'rnd_lens max':>14}{'rnd_iso max':>13}{'beats all':>11}")

    summary = {}
    for s in STRENGTHS:
        cand = results[f"{s}|topk"]["strict_single"]["acc"]
        nxt = results[f"{s}|next_k"]["strict_single"]["acc"]
        rl = [results[k]["strict_single"]["acc"] for k in results if k.startswith(f"{s}|random_lens")]
        ri = [results[k]["strict_single"]["acc"] for k in results if k.startswith(f"{s}|random_iso")]
        drop, rel = clean - cand, (clean - cand) / clean if clean else 0.0
        beats_lens = cand < min(rl) if rl else None
        beats_iso = cand < min(ri) if ri else None
        beats = bool(beats_lens) and bool(beats_iso)
        # p is reported PER NULL, not pooled. random_lens draws from the lens
        # dictionary and random_iso draws isotropic directions; they are
        # different null distributions and are not exchangeable with each other,
        # so 1/(19+19+1) would not be a valid combined p-value.
        summary[s] = {"candidate": cand, "drop": drop, "rel_reduction": rel,
                      "next_k": nxt, "random_lens": rl, "random_iso": ri,
                      "beats_all_random_lens": beats_lens,
                      "beats_all_random_iso": beats_iso,
                      "beats_all_random": beats,
                      "p_vs_random_lens": 1 / (len(rl) + 1) if beats_lens else None,
                      "p_vs_random_iso": 1 / (len(ri) + 1) if beats_iso else None}
        print(f"{s:<14}{cand:>9.1%}{drop:>8.1%}{rel:>8.1%}{nxt:>9.1%}"
              f"{(max(rl) if rl else float('nan')):>14.1%}"
              f"{(max(ri) if ri else float('nan')):>13.1%}{str(beats):>11}")

    h = summary[PRIMARY]
    lo = summary["light"]
    print("\n--- prereg §4 criteria ---")
    c1 = h["rel_reduction"] >= REL_REDUCTION_REQUIRED
    c2 = (h["drop"] - lo["drop"]) * 100 >= DOSE_GAP_PP
    c3 = all(summary[s]["beats_all_random"] for s in STRENGTHS)
    intact = results.get(f"{PRIMARY}|topk", {}).get("intact", {})
    t1 = intact.get("top1_match")
    c4 = t1 is not None and t1 >= PRESERVED_TOP1
    print(f"  substantial degradation (>={REL_REDUCTION_REQUIRED:.0%} rel) : "
          f"{h['rel_reduction']:.1%}  {'PASS' if c1 else 'FAIL'}")
    print(f"  dose-response (heavy-light >= {DOSE_GAP_PP}pp)        : "
          f"{(h['drop']-lo['drop'])*100:+.1f}pp  {'PASS' if c2 else 'FAIL'}")
    print(f"  candidate beats every random draw               : {'PASS' if c3 else 'FAIL'}")
    print(f"  intact top-1 >= {PRESERVED_TOP1:.0%}                          : "
          + (f"{t1:.1%}  {'PASS' if c4 else 'FAIL'}" if t1 is not None else "n/a"))

    if t1 is not None:
        d = diagnose(intact, degrading_drop=h["drop"], preserved_top1=PRESERVED_TOP1)
        print(f"\n  selectivity verdict: {d['verdict']}")
        print(f"  {d['reading']}")

    print("\n--- band-start dose-response (prereg §3 + Amendment 003) ---")
    depth = STRENGTHS[PRIMARY][-1] + 4          # n_layers - 1 = 35 for Qwen3-8B
    rels = []
    for nm in START_ORDER:
        st = STRENGTHS[nm][0]
        rels.append(summary[nm]["rel_reduction"])
        print(f"  {nm:<13} L{st:>2}..{STRENGTHS[nm][-1]}  start {100*st/depth:>3.0f}% depth  "
              f"width {len(STRENGTHS[nm]):>2}  rel reduction {summary[nm]['rel_reduction']:>7.1%}")

    mono = all(rels[i] <= rels[i + 1] + 1e-9 for i in range(len(rels) - 1))
    spread = max(rels) - min(rels)
    print(f"\n  Amendment 003 predicted: heavy-late < heavy < heavy-early < heavy-paper")
    print(f"  monotonic increasing as start moves down : {mono}")
    print(f"  spread across the four starts            : {spread:.1%}")
    if mono and spread >= 0.10:
        print("  -> consistent with UNDER-ABLATION: the primary band leaves")
        print("     recoverable content below L20, as Stage B2's readout suggested.")
    elif spread < 0.10:
        print("  -> band start is NOT load-bearing. This is the stronger outcome:")
        print("     it can be stated across four starts spanning 69%->37% of depth.")
    else:
        print("  -> non-monotonic. Report the pattern; do not pick a start by result.")

    print("\n  width control: medium (L20-27) and heavy-late (L24-31) are both 8 layers")
    print(f"    medium {summary['medium']['rel_reduction']:.1%} vs "
          f"heavy-late {summary['heavy-late']['rel_reduction']:.1%}"
          "   <- isolates start position from width")

    print("\n--- light, against its pre-registered prediction (§2.1) ---")
    print(f"  light rel reduction {lo['rel_reduction']:.1%}. Prereg predicted light may "
          "show little effect\n  because 8 downstream layers can re-establish content. "
          "A flat curve is therefore\n  WEAK evidence against the workspace under this nesting.")

    (out / "_summary.json").write_text(json.dumps(
        {"clean": clean, "strengths": summary,
         "criteria": {"substantial": c1, "dose_response": c2,
                      "beats_random": c3, "intact": c4},
         "band_start_dose_response": {
             "order": list(START_ORDER),
             "rel_reductions": rels,
             "monotonic": mono,
             "spread": spread},
         "amendments": ["002 strict on 73 single-token items",
                        "003 heavy-paper L13-31 added"],
         "config": vars(args)}, indent=2))
    print(f"\nwrote {out/'_summary.json'}")
    print("\nThe pass/partial/fail call is the user's (prereg §4). "
          "Log it as a gate decision (G0).")


if __name__ == "__main__":
    main()

## Cell 6 — Run Control A

~1 hour. Prints each condition as it completes.

**Resumable.** If the session dies, just re-run this cell — completed conditions are skipped.

In [ ]:
!python run_control_a.py \
    --model Qwen/Qwen3-8B \
    --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --out results/raw/controlA_qwen3-8b/ \
    --dtype bfloat16 \
    --n-draws 19 \
    --intact-passages 20

## Cell 7 — Dose-response plot

Candidate against the two random-null distributions, at each strength.

In [ ]:
import json, matplotlib.pyplot as plt

S = json.load(open("results/raw/controlA_qwen3-8b/_summary.json"))
clean = S["clean"]; order = ["light","medium","heavy","heavy-early","heavy-late"]
fig, ax = plt.subplots(figsize=(9,5))
for i,s in enumerate(order):
    d = S["strengths"][s]
    ax.scatter([i]*len(d["random_lens"]), d["random_lens"], c="tab:gray",
               alpha=.45, s=18, label="random (lens)" if i==0 else None)
    ax.scatter([i+.12]*len(d["random_iso"]), d["random_iso"], c="tab:olive",
               alpha=.45, s=18, marker="^", label="random (isotropic)" if i==0 else None)
    ax.scatter([i], [d["candidate"]], c="tab:red", s=110, zorder=5,
               label="candidate (top-k J-lens)" if i==0 else None)
    ax.scatter([i], [d["next_k"]], c="tab:blue", s=60, marker="s", zorder=4,
               label="next-k (complementary)" if i==0 else None)
ax.axhline(clean, ls="--", c="k", lw=1, label=f"clean ({clean:.1%})")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_ylabel("strict accuracy, single-token subset (n=73)")
ax.set_title("Control A — ablation vs matched random baselines")
ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/raw/controlA_qwen3-8b/dose_response.png", dpi=130)
plt.show()

print("Candidate BELOW every grey and olive point at a strength = that subspace is special.")
print("Candidate inside the cloud = ablating any subspace of this size does the same.")

## Cell 8 — Download

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("controlA_qwen3-8b", "zip", "results/raw/controlA_qwen3-8b")
files.download("controlA_qwen3-8b.zip")

---
## Report back

Paste the cell 6 summary block — the strength table, the four criteria, the selectivity verdict, and the band-sensitivity section — and attach the plot.

**The pass / partial / fail call is yours** (prereg §4, AI guide §2 row 12). I'll lay out where the data falls against each threshold and what a sceptical reviewer would push on; the call and the G0 gate entry are yours to make.

### Two things to hold in mind when reading it

`light` may show little effect. That is **pre-registered** (prereg §2.1): under shared-start nesting, ablating L20–23 leaves eight downstream layers to re-establish content. A flat curve is weak evidence against the workspace under this nesting, not strong.

If the intact side reads ~1.0 *and* nothing degrades, that is not a preserved intact side — it is an ablation that did nothing. `diagnose()` reports it as NO EFFECT, and the correct response is to check the hooks, not to write it up.